# Parcial 2C 2026

Este archivo contiene los ejercicios 2 y 3. Las consignas completas y todos los parámetros se encuentran en el enunciado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sig

from scipy.fft import dct
from scipy.special import logsumexp

np.set_printoptions(precision=3, suppress=True)

## Ejercicio 2

In [ ]:
pi = np.array([1.0, 0.0, 0.0])
A = np.array([
    [0.55, 0.45, 0.00],
    [0.00, 0.55, 0.45],
    [0.00, 0.00, 1.00],
])
logB = np.array([
    [-3.83,  0.00, -0.82],
    [ 0.00, -2.54, -1.74],
    [-0.81, -1.68,  0.00],
    [-1.01, -1.22,  0.00],
])

def log_probabilidades(P):
    P = np.asarray(P, dtype=float)
    logP = np.full(P.shape, -np.inf)
    mascara = P > 0
    logP[mascara] = np.log(P[mascara])
    return logP

In [ ]:
def log_forward(logB, pi, A):
    ### COMPLETAR LA FUNCIÓN

def log_backward(logB, A):
    T, N = logB.shape
    log_beta = np.full((T, N), -np.inf)
    log_A = log_probabilidades(A)
    log_beta[-1] = 0.0
    for t in range(T - 2, -1, -1):
        for i in range(N):
            log_beta[t, i] = ### <-- COMPLETAR
    return log_beta

In [ ]:
logp = ### <-- COMPLETAR
log_alpha = ### <-- COMPLETAR
log_beta = ### <-- COMPLETAR
gamma = ### <-- COMPLETAR
camino_local = ### <-- COMPLETAR

print('log p(Y|lambda):', logp)
print('gamma:\n', gamma)
print('Camino local (estados 1-3):', ...) ### <-- COMPLETAR

def camino_es_legal(camino, A):
    return ... ### <-- COMPLETAR (revisar que cada transicion consecutiva tenga probabilidad positiva)

legal_local = ### <-- COMPLETAR (evaluar la legalidad del camino local)
print('¿El camino local es legal?', ...) ### <-- COMPLETAR

In [ ]:
def log_viterbi(logB, pi, A):
    T, N = logB.shape
    log_delta = np.full((T, N), -np.inf)
    psi = np.zeros((T, N), dtype=int)
    log_A = log_probabilidades(A)
    log_delta[0] = log_probabilidades(pi) + logB[0]

    for t in range(1, T):
        for j in range(N):
            candidatos = log_delta[t - 1] + log_A[:, j]
            psi[t, j] = np.argmax(candidatos)
            log_delta[t, j] = candidatos[psi[t, j]] + logB[t, j]

    path = np.zeros(T, dtype=int)
    path[-1] = ### <-- COMPLETAR (ayuda: no poner exactamente lo mismo que en la solución de clase)
    for t in range(T - 2, -1, -1):
        path[t] = psi[t + 1, path[t + 1]]
    score = log_delta[-1, ...] ### <-- COMPLETAR (ayuda: ídem comentario anterior, leer el enunaciado)
    return score, path, log_delta, psi

# Llamar a la funcion y mostrar el camino usando estados 1-3
score_viterbi, camino_viterbi, log_delta, psi = ### <-- COMPLETAR
print(f'Score Viterbi: {score_viterbi:.6f}')
print('Camino Viterbi (estados 1-3):', ...) ### <-- COMPLETAR
print('Forward >= Viterbi:', ...) ### <-- COMPLETAR

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for j in range(gamma.shape[1]):
    ax.plot(np.arange(1, len(gamma) + 1), gamma[:, j], marker='o', label=f'Estado {j+1}')
ax.set_xlabel('Frame $t$')
ax.set_ylabel('$\\gamma_t(j)$')
ax.set_xticks(np.arange(1, len(gamma) + 1))
ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

### Interpretación del ejercicio 2

<mark>Completar<mark>

## Ejercicio 3

In [ ]:
fs = 16000
F0 = 125
duracion_segmento = 0.36
formantes = {
    'a': [750, 1300, 2500],
    'i': [300, 2300, 3000],
}
bws = [80, 90, 120]
L = 400
H = 160
N_FFT = 512
M = 26
C = 13

def sintetizar_vocal(formantes_hz, duracion, fs=fs, F0=F0, bws=bws):
    T = int(round(duracion * fs))
    fuente = np.zeros(T)
    periodo = int(round(fs / F0))
    fuente[::periodo] = 1.0
    salida = fuente.copy()
    for f, bw in zip(formantes_hz, bws):
        r = np.exp(-np.pi * bw / fs)
        theta = 2 * np.pi * f / fs
        b = [1 - r * r]
        a = [1, -2 * r * np.cos(theta), r * r]
        salida = sig.lfilter(b, a, salida)
    return salida / np.max(np.abs(salida))

def hz_a_mel(f):
    return 2595 * np.log10(1 + np.asarray(f) / 700)

def mel_a_hz(m):
    return 700 * (10 ** (np.asarray(m) / 2595) - 1)

def banco_mel(n_mels=M, n_fft=N_FFT, fs=fs):
    puntos_mel = np.linspace(hz_a_mel(0), hz_a_mel(fs / 2), n_mels + 2)
    puntos_hz = mel_a_hz(puntos_mel)
    bins = np.floor((n_fft + 1) * puntos_hz / fs).astype(int)
    bins = np.clip(bins, 0, n_fft // 2)
    B = np.zeros((n_mels, n_fft // 2 + 1))
    for m in range(1, n_mels + 1):
        izq, centro, der = bins[m - 1], bins[m], bins[m + 1]
        if centro > izq:
            B[m - 1, izq:centro] = (np.arange(izq, centro) - izq) / (centro - izq)
        if der > centro:
            B[m - 1, centro:der] = (der - np.arange(centro, der)) / (der - centro)
    return B

def mfcc_propio(x):
    T = 1 + (len(x) - L) // H
    indices = np.arange(L)[None, :] + H * np.arange(T)[:, None]
    frames = x[indices] * np.hamming(L)[None, :]
    X = np.fft.rfft(frames, n=N_FFT, axis=1)
    S = np.abs(X) ** 2
    E = S @ banco_mel().T
    logmel = np.log(E + 1e-10)
    mfcc = dct(logmel, type=2, axis=1, norm='ortho')[:, :C]
    tiempos = (np.arange(T) * H + L / 2) / fs
    return mfcc, tiempos

In [ ]:
# Sintetizar /a/ e /i/, concatenarlas y calcular los MFCCs.
x_a = ### <-- COMPLETAR
x_i = ### <-- COMPLETAR
x = ### <-- COMPLETAR
mfcc, tiempos = ### <-- COMPLETAR
print('Dimensiones de la matriz de MFCCs:', mfcc.shape)

In [1]:
def calcular_deltas(C_mfcc, theta=2):
    C_pad = np.pad(C_mfcc, ((theta, theta), (0, 0)), mode='edge')
    numerador = np.zeros_like(C_mfcc)
    for tau in range(1, theta + 1):
        numerador += ### <-- COMPLETAR
    denominador = ### <-- COMPLETAR
    return numerador / denominador
    
# Obtener la norma por frame y máximo restringido a 0.20-0.52
deltas = ### <-- COMPLETAR
norma_delta = ### <-- COMPLETAR
zona = (tiempos >= ...) & (tiempos <= ...) ### <-- COMPLETAR
indices_zona = np.flatnonzero(zona)
indice_transicion = indices_zona[...] ### <-- COMPLETAR
print(f'Tiempo estimado de transición: {tiempos[indice_transicion]:.4f} s')
print(f'Frame de Python {indice_transicion} e índice matemático t={indice_transicion + 1}')

SyntaxError: invalid syntax (507613188.py, line 5)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=False)
lim = np.max(np.abs(mfcc))
im = axes[0].imshow(mfcc.T, origin='lower', aspect='auto', extent=[tiempos[0], tiempos[-1], 0, C - 1], cmap='coolwarm', vmin=-lim, vmax=lim)
axes[0].axvline(duracion_segmento, color='black', ls='--', label='Transición real')
axes[0].set_ylabel('Coeficiente $d$')
axes[0].set_title('MFCC $c_d[t]$')
axes[0].legend(loc='upper right')
fig.colorbar(im, ax=axes[0], label='Valor del MFCC')

axes[1].plot(...) ### <-- COMPLETAR
axes[1].axvline(duracion_segmento, color='black', ls='--', label='Transición real')
axes[1].axvline(tiempos[indice_transicion], color='tab:red', ls=':', label='Máximo detectado')
axes[1].set_xlabel('Tiempo del centro del frame (s)')
axes[1].set_ylabel('$\\|\\Delta \\mathbf{c}[t]\\|_2$')
axes[1].set_title('Cambio local de los MFCC')
axes[1].grid(alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.show()

### Interpretación del ejercicio 3

<mark>Completar<mark>